# 04 — Demo de entrenamiento CNN (1 fold)

Ejecuta un único fold LOSO sobre datos sintéticos o un subset real para validar:
- Que el forward/backward funcionan.
- Que las curvas de loss bajan.
- Que early stopping se dispara correctamente.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append('..')
import torch, numpy as np
from torch.utils.data import DataLoader, TensorDataset
from src.models.cnn import ModSpecCNN
from src.train import TrainConfig, train_cnn

rng = np.random.default_rng(0)
n = 500
Xtr = rng.standard_normal((n, 19, 45, 45)).astype('float32')
ytr = (rng.random(n) > 0.5).astype('int64')
Xv = rng.standard_normal((100, 19, 45, 45)).astype('float32')
yv = (rng.random(100) > 0.5).astype('int64')

class Wrap(torch.utils.data.Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return torch.from_numpy(self.X[i]), torch.tensor(int(self.y[i])), 's'
tr = DataLoader(Wrap(Xtr, ytr), batch_size=4, shuffle=True)
vl = DataLoader(Wrap(Xv, yv), batch_size=32)
model = ModSpecCNN()
out = train_cnn(model, tr, vl, TrainConfig(epochs=5), train_labels=ytr)
out['history']